In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.business_transformation.emi_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. CONFIGURATION
# ============================================================
source_table = "fintech360.business_transformation.emi_data"
target_table = "fintech360.gold.emi_data"

spark.sql("CREATE SCHEMA IF NOT EXISTS fintech360.gold")

# ============================================================
# 2. READ BUSINESS-TRANSFORMED DATA
# ============================================================
df = spark.read.table(source_table)

# ============================================================
# 3. DEDUPLICATE THE INCOMING BATCH
# Keep the latest transformed record for each EMI
# ============================================================
dedupe_window = Window.partitionBy("emi_id").orderBy(
    F.col("business_transformed_at").desc_nulls_last(),
    F.col("updated_at").cast("long").desc_nulls_last()
)

df = (
    df.withColumn("_row_num", F.row_number().over(dedupe_window))
      .filter(F.col("_row_num") == 1)
      .drop("_row_num")
)

# ============================================================
# 4. STANDARDIZE CORE EMI FIELDS
# ============================================================
df = (
    df
    .withColumn("gold_emi_id", F.col("emi_id").cast("long"))
    .withColumn("gold_loan_id", F.col("loan_id").cast("long"))
    .withColumn("gold_installment_number", F.col("business_installment_number").cast("int"))
    .withColumn("gold_emi_status", F.upper(F.trim(F.col("business_emi_status"))))
    .withColumn("gold_payment_method", F.upper(F.trim(F.col("business_payment_method"))))
    .withColumn("gold_mandate_type", F.upper(F.trim(F.col("business_mandate_type"))))
    .withColumn("gold_emi_amount", F.col("business_emi_amount").cast("decimal(18,2)"))
    .withColumn("gold_paid_amount", F.col("business_paid_amount").cast("decimal(18,2)"))
    .withColumn("gold_late_fee_amount", F.col("business_late_fee").cast("decimal(18,2)"))
    .withColumn("gold_dpd", F.col("business_dpd").cast("int"))
)

# ============================================================
# 5. DUE DATE, PAYMENT DATE AND REPORTING DIMENSIONS
# Timestamps in the raw columns are Unix seconds.
# ============================================================
df = (
    df
    .withColumn(
        "gold_due_timestamp",
        F.coalesce(
            F.to_timestamp(F.col("business_due_date_datetime")),
            F.to_timestamp(F.from_unixtime(F.col("due_date").cast("long")))
        )
    )
    .withColumn(
        "gold_paid_timestamp",
        F.coalesce(
            F.to_timestamp(F.col("business_paid_at_datetime")),
            F.to_timestamp(F.from_unixtime(F.col("paid_at").cast("long")))
        )
    )
    .withColumn("gold_due_date", F.to_date("gold_due_timestamp"))
    .withColumn("gold_paid_date", F.to_date("gold_paid_timestamp"))
    .withColumn("gold_due_month", F.date_format("gold_due_timestamp", "yyyy-MM"))
    .withColumn("gold_due_year", F.year("gold_due_timestamp"))
    .withColumn("gold_due_day", F.dayofmonth("gold_due_timestamp"))
    .withColumn("gold_due_day_of_week", F.date_format("gold_due_timestamp", "EEEE"))
    .withColumn("gold_paid_month", F.date_format("gold_paid_timestamp", "yyyy-MM"))
)

# ============================================================
# 6. OUTSTANDING AMOUNT AND REPAYMENT PERCENTAGE
# ============================================================
df = (
    df
    .withColumn(
        "gold_outstanding_amount",
        F.greatest(
            F.coalesce(F.col("gold_emi_amount"), F.lit(0).cast("decimal(18,2)")) -
            F.coalesce(F.col("gold_paid_amount"), F.lit(0).cast("decimal(18,2)")),
            F.lit(0).cast("decimal(18,2)")
        )
    )
    .withColumn(
        "gold_repayment_percentage",
        F.when(
            F.col("gold_emi_amount") > 0,
            F.round(
                F.col("gold_paid_amount") * 100.0 / F.col("gold_emi_amount"),
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "gold_collection_gap",
        F.col("gold_outstanding_amount")
    )
)

# ============================================================
# 7. REPAYMENT STATUS FLAGS
# ============================================================
df = (
    df
    .withColumn(
        "gold_fully_paid_flag",
        F.when(
            (F.col("gold_emi_amount") > 0) &
            (F.col("gold_outstanding_amount") == 0),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_partially_paid_flag",
        F.when(
            (F.col("gold_paid_amount") > 0) &
            (F.col("gold_outstanding_amount") > 0),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_unpaid_flag",
        F.when(
            (F.col("gold_paid_amount") <= 0) &
            (F.col("gold_outstanding_amount") > 0),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_overdue_flag",
        F.when(
            (F.col("gold_outstanding_amount") > 0) &
            (F.col("gold_due_date") < F.current_date()),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_pending_flag",
        F.when(
            (F.col("gold_outstanding_amount") > 0) &
            (F.col("gold_due_date") >= F.current_date()),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 8. DELINQUENCY AND OVERDUE EXPOSURE
# ============================================================
df = (
    df
    .withColumn(
        "gold_dpd_bucket",
        F.when(F.col("gold_dpd").isNull(), "UNKNOWN")
         .when(F.col("gold_dpd") <= 0, "CURRENT")
         .when(F.col("gold_dpd") <= 30, "01_1_TO_30_DPD")
         .when(F.col("gold_dpd") <= 60, "02_31_TO_60_DPD")
         .when(F.col("gold_dpd") <= 90, "03_61_TO_90_DPD")
         .otherwise("04_90_PLUS_DPD")
    )
    .withColumn(
        "gold_overdue_emi_amount",
        F.when(
            F.col("gold_overdue_flag") == 1,
            F.col("gold_outstanding_amount")
        ).otherwise(F.lit(0).cast("decimal(18,2)"))
    )
    .withColumn(
        "gold_overdue_late_fee_amount",
        F.when(
            F.col("gold_overdue_flag") == 1,
            F.coalesce(F.col("gold_late_fee_amount"), F.lit(0).cast("decimal(18,2)"))
        ).otherwise(F.lit(0).cast("decimal(18,2)"))
    )
    .withColumn(
        "gold_total_overdue_exposure",
        F.col("gold_overdue_emi_amount") +
        F.col("gold_overdue_late_fee_amount")
    )
    .withColumn(
        "gold_high_delinquency_flag",
        F.when(
            (F.col("gold_overdue_flag") == 1) &
            (F.col("gold_dpd") > 90),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_high_late_fee_flag",
        F.when(
            (F.col("gold_overdue_flag") == 1) &
            (F.col("gold_late_fee_amount") >= 500),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 9. PAYMENT TIMELINESS
# Positive delay means payment happened after the due date.
# Unpaid EMIs have no actual payment delay yet.
# ============================================================
df = (
    df
    .withColumn(
        "gold_payment_delay_days",
        F.when(
            F.col("gold_paid_date").isNotNull() &
            F.col("gold_due_date").isNotNull(),
            F.datediff(F.col("gold_paid_date"), F.col("gold_due_date"))
        )
    )
    .withColumn(
        "gold_paid_late_flag",
        F.when(F.col("gold_payment_delay_days") > 0, 1).otherwise(0)
    )
    .withColumn(
        "gold_payment_timeliness",
        F.when(F.col("gold_paid_date").isNull(), "NOT_PAID")
         .when(F.col("gold_payment_delay_days") > 0, "PAID_AFTER_DUE_DATE")
         .otherwise("PAID_ON_OR_BEFORE_DUE_DATE")
    )
    .withColumn(
        "gold_late_fee_to_emi_percentage",
        F.when(
            F.col("gold_emi_amount") > 0,
            F.round(
                F.col("gold_late_fee_amount") * 100.0 / F.col("gold_emi_amount"),
                2
            )
        ).otherwise(F.lit(0.0))
    )
)

# ============================================================
# 10. INSTALLMENT AND PAYMENT CHANNEL SEGMENTS
# ============================================================
df = (
    df
    .withColumn(
        "gold_installment_stage",
        F.when(F.col("gold_installment_number").isNull(), "UNKNOWN")
         .when(F.col("gold_installment_number") <= 3, "EARLY_TENURE")
         .when(F.col("gold_installment_number") <= 6, "MID_TENURE")
         .otherwise("LATE_TENURE")
    )
    .withColumn(
        "gold_payment_channel_group",
        F.when(F.col("gold_payment_method").isin("NACH", "AUTO_DEBIT", "AUTOPAY"), "AUTO_DEBIT")
         .when(F.col("gold_payment_method").isin("UPI"), "UPI")
         .when(F.col("gold_payment_method").isin("CARD", "CREDIT_CARD", "DEBIT_CARD"), "CARD")
         .when(F.col("gold_payment_method").isin("CASH"), "CASH")
         .when(F.col("gold_payment_method").isin("BANK_TRANSFER", "NEFT", "IMPS"), "BANK_PAYMENT")
         .when(F.col("gold_payment_method").isNull(), "UNKNOWN")
         .otherwise("OTHER_PAYMENT")
    )
    .withColumn(
        "gold_autopay_mandate_flag",
        F.when(
            F.upper(F.trim(F.col("gold_mandate_type"))).isin(
                "NACH", "UPI_AUTOPAY", "CARD_AUTOPAY", "E_NACH"
            ),
            1
        ).otherwise(0)
    )
)

# ============================================================
# 11. REVIEW SIGNALS AND DATA QUALITY
# ============================================================
df = (
    df
    .withColumn(
        "gold_review_signal",
        F.when(F.col("gold_emi_id").isNull(), "MISSING_EMI_ID")
         .when(F.col("gold_loan_id").isNull(), "MISSING_LOAN_ID")
         .when(F.col("gold_emi_amount").isNull() | (F.col("gold_emi_amount") < 0), "INVALID_EMI_AMOUNT")
         .when(F.col("gold_paid_amount").isNull() | (F.col("gold_paid_amount") < 0), "INVALID_PAID_AMOUNT")
         .when(F.col("gold_paid_amount") > F.col("gold_emi_amount"), "PAID_AMOUNT_EXCEEDS_EMI")
         .when(F.col("gold_overdue_flag") == 1, "OVERDUE_EMI")
         .when(F.col("gold_paid_late_flag") == 1, "PAID_LATE")
         .when(F.col("gold_partially_paid_flag") == 1, "PARTIAL_REPAYMENT")
         .when(F.col("gold_pending_flag") == 1, "EMI_PENDING")
         .otherwise("NO_BASIC_REVIEW_SIGNAL")
    )
    .withColumn(
        "gold_data_quality_status",
        F.when(F.col("gold_emi_id").isNull(), "INVALID")
         .when(F.col("gold_loan_id").isNull(), "INVALID")
         .when(
             F.col("gold_emi_amount").isNull() |
             (F.col("gold_emi_amount") < 0),
             "INVALID"
         )
         .when(
             F.col("gold_paid_amount").isNull() |
             (F.col("gold_paid_amount") < 0) |
             (F.col("gold_paid_amount") > F.col("gold_emi_amount")),
             "INVALID"
         )
         .otherwise("VALID")
    )
)

# ============================================================
# 12. GOLD METADATA
# ============================================================
df = (
    df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_table", F.lit(source_table))
)

gold_df = df

# ============================================================
# 13. APPEND TO GOLD TABLE
# Skip EMI IDs already present for normal sequential reruns.
# ============================================================
if spark.catalog.tableExists(target_table):
    existing_ids = (
        spark.read.table(target_table)
        .select(F.col("emi_id").alias("_existing_emi_id"))
        .filter(F.col("_existing_emi_id").isNotNull())
        .dropDuplicates()
    )

    gold_df = gold_df.join(
        existing_ids,
        gold_df["emi_id"] == existing_ids["_existing_emi_id"],
        "left_anti"
    )

gold_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)

print(f"Gold data appended successfully to {target_table}")

# ============================================================
# 14. VALIDATE GOLD TABLE
# ============================================================
spark.sql(f"""
SELECT
    COUNT(*) AS total_emi_records,
    COUNT(DISTINCT emi_id) AS unique_emi_records,
    COUNT(DISTINCT loan_id) AS unique_loans,
    SUM(gold_fully_paid_flag) AS fully_paid_emis,
    SUM(gold_partially_paid_flag) AS partially_paid_emis,
    SUM(gold_unpaid_flag) AS unpaid_emis,
    SUM(gold_overdue_flag) AS overdue_emis,
    ROUND(SUM(gold_outstanding_amount), 2) AS total_outstanding_amount,
    ROUND(SUM(gold_total_overdue_exposure), 2) AS total_overdue_exposure,
    SUM(gold_paid_late_flag) AS paid_late_emis,
    SUM(gold_high_delinquency_flag) AS high_delinquency_emis,
    SUM(CASE WHEN gold_data_quality_status = 'INVALID' THEN 1 ELSE 0 END) AS invalid_records
FROM {target_table}
""").show(truncate=False)

# Monthly collection and overdue summary
spark.sql(f"""
SELECT
    gold_due_month,
    COUNT(*) AS total_emis,
    SUM(gold_fully_paid_flag) AS fully_paid_emis,
    SUM(gold_overdue_flag) AS overdue_emis,
    ROUND(SUM(gold_paid_amount), 2) AS total_paid_amount,
    ROUND(SUM(gold_outstanding_amount), 2) AS total_outstanding_amount,
    ROUND(SUM(gold_total_overdue_exposure), 2) AS total_overdue_exposure
FROM {target_table}
GROUP BY gold_due_month
ORDER BY gold_due_month
""").show(100, truncate=False)

# Preview
spark.read.table(target_table).show(10, truncate=False)
